# Flujo NLI

El objetivo es clasificar la relación entre premisa e hipótesis. Ejecuta las dos primeras celdas y después el componente que quieras demostrar. La evaluación utiliza el código de Aitor. El cuaderno anterior conserva los resultados históricos.

In [ ]:
from pathlib import Path
from importlib.metadata import version

from nli_project.data.paths import repository_root

PROJECT = repository_root()

import pandas as pd
from IPython.display import display
from nli_project.translation.ollama import LANGUAGES, OllamaTranslator, load_translations, translate_dataframe, translate_file
from nli_project.models.classifiers import CLASSIFIER_NAMES
from nli_project.preprocessing.parameter_variants import load_variant_dataset, materialize_variants
from nli_project.preprocessing.preprocesser import NLITextPreprocesser
from nli_project.preprocessing.preprocessing import ENGLISH_PREPROCESSING_PARAMS, MULTILINGUAL_PREPROCESSING_PARAMS, preprocess_partition
from experiments.scripts.run_experiment_matrix import DEFAULT_CLASSIFIERS, DEFAULT_REPRESENTATIONS, run_matrix
from nli_project.data.dataset import (
    LABEL_NAMES,
    make_split,
)
from nli_project.training.training import build_pipeline
from nli_project.features.vectorizer import NLITextVectorizer

MODE = "smoke"  # "full" usa todos los datos de desarrollo y CatBoost.
RUN_EXPERIMENTS = True
SHOW_HOLDOUT = True
USE_TRANSLATIONS = True
COMPARE_TOKENIZERS = False  # Espacios frente a spaCy, con texto original.
TUNE_MINILM = False  # Rejilla de spaCy, chi-cuadrado y profundidad.
RUN_MINILM = False  # Comparacion completa con MLP y XGBoost.
TRY_OLLAMA = False  # Solo traduce el ejemplo de demostración.
GENERATE_FULL_TRANSLATIONS = False  # Requiere MODE = "full" y Ollama local.
MODEL = "gemma3:4b"
DEMO_CLASSIFIER = "naive_bayes"
SEED = 42
ARTIFACTS = PROJECT / "experiments/.local/workflow_javi_aitor"
HISTORICAL = PROJECT / "experiments/results/validation/workflow_javi_aitor"
TRAIN_CSV = PROJECT / "experiments/data/train.csv"
translation_name = f"train_{MODEL.replace(':', '_').replace('/', '_')}.csv"
TRANSLATION_CSV = PROJECT / "experiments/results/translation/corpus" / translation_name
if not TRANSLATION_CSV.is_file():
    TRANSLATION_CSV = PROJECT / "experiments/.local/translations" / translation_name

representations = {
    "pair_diff": {**DEFAULT_REPRESENTATIONS["pair_diff"], "min_df": 2},
    "char_tfidf": {"method": "char_tfidf", "min_df": 2, "max_features": 10000},
}
classifiers = ["naive_bayes"] if MODE == "smoke" else ["naive_bayes", "catboost"]
aux_features = {"use_pos_features": False, "use_relation_features": False}
select_k = None if MODE == "smoke" else 300

if MODE not in {"smoke", "full"}:
    raise ValueError("MODE debe ser 'smoke' o 'full'.")
print(PROJECT, "· modo", MODE)
for package in ("pandas", "scikit-learn", "spacy", "en-core-web-sm"):
    print(package, version(package))

## Datos y particiones

Reservamos validación agrupando por premisa original. En modo rápido se toma una muestra por clase de cada partición. El test de la competición sirve aquí para comprobar el esquema.

In [ ]:
options = {"dtype": {"id": str}, "keep_default_na": False}
source = pd.read_csv(TRAIN_CSV, **options)
competition_test = pd.read_csv(PROJECT / "experiments/data/test.csv", **options)
for frame in (source, competition_test):
    assert frame.id.is_unique and frame.id.str.strip().ne("").all()
    assert set(frame.lang_abv) == {"en", *LANGUAGES}
assert set(source.label) == set(LABEL_NAMES) and "label" not in competition_test
assert set(source.id).isdisjoint(competition_test.id)

train, val = make_split(source, seed=SEED, group_by_premise=True)
if MODE == "smoke":
    train = train.sample(frac=1, random_state=SEED).groupby("label").head(35).reset_index(drop=True)
    val = val.sample(frac=1, random_state=SEED).groupby("label").head(12).reset_index(drop=True)
assert set(train.id).isdisjoint(val.id)
assert set(train.premise).isdisjoint(val.premise)
print("Ajuste:", len(train), "· validación:", len(val), "· test sin etiqueta:", len(competition_test))
display(pd.crosstab(train.lang_abv, train.label))

## Preprocesador

La receta neutral conserva los 15 idiomas, usa espacios y mantiene los acentos. Para idiomas sin espacios, los n-gramas de caracteres aportan una segunda representación. La receta inglesa filtra filas inglesas y permite lematización y POS. Alba carga el modelo local de spaCy al construir el preprocesador, incluso con la receta neutral.

In [ ]:
examples = source.groupby("lang_abv").head(1)
for name, params in (("neutral", MULTILINGUAL_PREPROCESSING_PARAMS),
                     ("inglés", ENGLISH_PREPROCESSING_PARAMS)):
    example = NLITextPreprocesser(**params).preprocess_dataset(examples)
    print(name, "· filas retenidas:", len(example))
    display(example[["lang_abv", "premise", "premise_processed"]].head(4))

## Traducción por lotes

El ejemplo usa una respuesta de prueba por defecto. Ollama recibe pares de un idioma por lote y devuelve los textos vinculados al ID. La traducción añade columnas; el idioma de origen permanece disponible para el análisis.

In [ ]:
from unittest.mock import Mock

example = pd.DataFrame([
    {"id": "001", "lang_abv": "es", "premise": "No hay café.", "hypothesis": "Hay café."},
    {"id": "002", "lang_abv": "en", "premise": "There is tea.", "hypothesis": "There is a drink."},
])
if TRY_OLLAMA:
    client = OllamaTranslator(MODEL)
    print("Ejemplo con Ollama local")
else:
    client = Mock()
    client.translate_batch.return_value = {
        "001": {"premise": "There is no coffee.", "hypothesis": "There is coffee."},
    }
    print("Ejemplo simulado · Ollama omitido")
demo_translation = translate_dataframe(example, client, batch_size=2)
pd.testing.assert_frame_equal(demo_translation[example.columns], example)
display(demo_translation)

## Permutador

Cada variante publica un DataFrame y parámetros consultables en su manifiesto. Usamos los metadatos devueltos para cargar el pickle.

In [ ]:
permutations = materialize_variants(
    train, ARTIFACTS / "variants",
    base_params=MULTILINGUAL_PREPROCESSING_PARAMS,
    variants=[{"name": "neutral", "params": {}},
              {"name": "neutral_con_mayúsculas", "params": {"lowercase": False}}],
    cache_dir=ARTIFACTS / "preprocessed",
)
display(pd.DataFrame(permutations)[["name", "variant_id", "n_rows"]])
display(load_variant_dataset(permutations[0])[["id", "premise_processed"]].head(3))

## Vectorizador y modelos

`pair_diff` usa bloques para premisa, hipótesis y diferencia, además de señales de solapamiento. `char_tfidf` representa fragmentos de caracteres. Ambos ajustan su vocabulario en desarrollo. La fábrica construye modelos a partir de su nombre.

In [ ]:
vector_demo = preprocess_partition(train.head(15), MULTILINGUAL_PREPROCESSING_PARAMS)
for name, params in representations.items():
    vectorizer = NLITextVectorizer(
        columns=("premise_processed", "hypothesis_processed"), **params
    )
    print(name, vectorizer.fit_transform(vector_demo).shape)
print("Representaciones disponibles:", ", ".join(sorted(NLITextVectorizer.VALID_METHODS)))

In [ ]:
print("Clasificadores disponibles:", ", ".join(CLASSIFIER_NAMES))
demo_train = preprocess_partition(train.head(30), MULTILINGUAL_PREPROCESSING_PARAMS)
demo_val = preprocess_partition(val.head(6), MULTILINGUAL_PREPROCESSING_PARAMS)
pipeline = build_pipeline(
    representations["pair_diff"], classifier=DEMO_CLASSIFIER,
    classifier_params=DEFAULT_CLASSIFIERS.get(DEMO_CLASSIFIER), random_state=SEED,
    select_k=None, **aux_features,
)
pipeline.fit(demo_train, demo_train.label)
print(DEMO_CLASSIFIER, "| pasos:", list(pipeline.named_steps))
display(demo_val[["id", "label"]].assign(prediction=pipeline.predict(demo_val)))

## Comparación reproducible

Para medir el efecto de traducir, los dos brazos usan las mismas filas, etiquetas, premisas de agrupación, representaciones, clasificadores y receta neutral. Solo cambian los textos de entrada. POS y las reglas de relación, que incluyen negaciones inglesas, están desactivados en ambos brazos.

Si falta un CSV traducido, la comparación continúa con el original. Para generar el CSV completo, selecciona modo `full` y activa `GENERATE_FULL_TRANSLATIONS`. La matriz reutiliza ejecuciones completas con la misma configuración y los mismos datos.

In [ ]:
records = []
if not RUN_EXPERIMENTS:
    print("Experimentos omitidos por RUN_EXPERIMENTS.")
else:
    arms = {"original": (train, val, MULTILINGUAL_PREPROCESSING_PARAMS)}
    if USE_TRANSLATIONS:
        if not TRANSLATION_CSV.is_file() and MODE == "full" and GENERATE_FULL_TRANSLATIONS:
            translate_file(TRAIN_CSV, TRANSLATION_CSV, model=MODEL)
        translated = None
        if TRANSLATION_CSV.is_file():
            try:
                translated = load_translations(source, TRANSLATION_CSV).set_index("id")
            except (ValueError, AssertionError) as error:
                print(f"Brazo traducido omitido: CSV inválido ({error}).")
        else:
            print("Brazo traducido omitido: falta el CSV. Ollama no se ha llamado.")
        if translated is not None:
            arms["traducido"] = (
                translated.loc[train.id].reset_index(),
                translated.loc[val.id].reset_index(),
                {**MULTILINGUAL_PREPROCESSING_PARAMS,
                 "columns": ("premise_translated", "hypothesis_translated")},
            )
            print("Traducciones verificadas:", TRANSLATION_CSV)
    else:
        print("Brazo traducido omitido por USE_TRANSLATIONS.")

    train_paths, val_paths = [], []
    for arm, (arm_train, arm_val, params) in arms.items():
        partitions = [(arm_train, train_paths)]
        if SHOW_HOLDOUT:
            partitions.append((arm_val, val_paths))
        for frame, paths in partitions:
            info, = materialize_variants(
                frame, ARTIFACTS / "variants", base_params=params,
                variants=[{"name": "neutral", "params": {}}],
                cache_dir=ARTIFACTS / "preprocessed",
            )
            paths.append((arm, Path(info["dataset_path"])))

    records = run_matrix(
        train_paths, representations, classifiers, ARTIFACTS / MODE,
        n_splits=2 if MODE == "smoke" else 5, seed=SEED,
        select_k=select_k, group_by_premise=True, reuse=True,
        holdout_datasets=val_paths,
        holdout_classifiers=classifiers if SHOW_HOLDOUT else [],
        **aux_features,
    )
    display(pd.DataFrame([
        {"brazo": row["dataset"], "representación": row["representation"],
         "modelo": row["classifier"], "accuracy CV": row["summary"]["accuracy_mean"],
         "macro-F1 CV": row["summary"]["macro_f1_mean"],
         "ajuste CV (s)": row["summary"]["fit_seconds_total"],
         "predicción por fold (s)": row["summary"]["predict_seconds_mean"]}
        for row in records
    ]).round(4))

## Evaluación reservada

La celda presenta las métricas por configuración y muestra las matrices y errores de `pair_diff` con Naive Bayes en ambos brazos. La elección de este ejemplo es fija y evita escogerlo por sus resultados de validación.

In [ ]:
evaluated = [row for row in records if "holdout" in row]
if not evaluated:
    print("Evaluación omitida: no se solicitó validación o no se ejecutó la matriz.")
else:
    display(pd.DataFrame([
        {"brazo": row["dataset"], "representación": row["representation"],
         "modelo": row["classifier"], "accuracy": row["holdout"]["accuracy"],
         "macro-F1": row["holdout"]["macro_f1"], "tiempo (s)": row["holdout"]["runtime_seconds"]}
        for row in evaluated
    ]).round(4))
    for row in evaluated:
        if (row["classifier"], row["representation"]) != ("naive_bayes", "pair_diff"):
            continue
        metrics = row["holdout"]
        print(row["dataset"], "· filas = real, columnas = predicción")
        display(pd.DataFrame(metrics["confusion_matrix"],
                             index=LABEL_NAMES.values(), columns=LABEL_NAMES.values()))
        predictions = pd.read_csv(metrics["predictions_path"], dtype={"id": str})
        examples = arms[row["dataset"]][1]
        errors = predictions.loc[predictions.label.ne(predictions.prediction)].merge(
            examples.drop(columns=["label", "lang_abv"]), on="id", validate="one_to_one"
        )
        columns = ["id", "lang_abv", "premise", "hypothesis"]
        if row["dataset"] == "traducido":
            columns += ["premise_translated", "hypothesis_translated"]
        display(errors.assign(real=errors.label.map(LABEL_NAMES),
                              predicha=errors.prediction.map(LABEL_NAMES))
                      [columns + ["real", "predicha"]].head(5))

## Espacios frente a spaCy

Activa `COMPARE_TOKENIZERS` para comparar los dos tokenizadores con texto original y `pair_diff`. El lematizado, POS y las reglas de relación siguen desactivados. Los dos tokenizadores comparten modelos y folds.

In [ ]:
if COMPARE_TOKENIZERS:
    variants = materialize_variants(
        train, ARTIFACTS / "variants",
        base_params=MULTILINGUAL_PREPROCESSING_PARAMS,
        variants=[{"name": tokenizer, "params": {"tokenizer": tokenizer}}
                  for tokenizer in ("whitespace", "spacy")],
        cache_dir=ARTIFACTS / "preprocessed",
    )
    tokenizer_results = run_matrix(
        [(row["name"], Path(row["dataset_path"])) for row in variants],
        {"pair_diff": representations["pair_diff"]}, classifiers,
        ARTIFACTS / "tokenizers" / MODE,
        n_splits=2 if MODE == "smoke" else 5, seed=SEED,
        select_k=select_k, group_by_premise=True, reuse=True, **aux_features,
    )
    display(pd.DataFrame([
        {"tokenizador": row["dataset"], "modelo": row["classifier"],
         "accuracy": row["summary"]["accuracy_mean"],
         "macro-F1": row["summary"]["macro_f1_mean"]}
        for row in tokenizer_results
    ]).round(4))
else:
    print("Comparación de tokenizadores omitida por COMPARE_TOKENIZERS.")

## MiniLM con y sin traducción

Activa `RUN_MINILM` para ejecutar los cinco folds agrupados con MLP y XGBoost. Requiere las traducciones guardadas, pesos locales de MiniLM y CUDA para XGBoost. La celda muestra los resultados guardados si ya existen.

In [ ]:
import json

minilm_output = ARTIFACTS.parent / f"minilm_translation_run{SEED}"
archived_minilm = PROJECT / "experiments/results/translation" / f"minilm_seed{SEED}"
result_path = minilm_output / "results.json"
if not RUN_MINILM and not result_path.is_file():
    result_path = archived_minilm / "results.json"
if RUN_MINILM and not result_path.is_file():
    from experiments.scripts.run_minilm_translation import run

    if not TRANSLATION_CSV.is_file():
        raise FileNotFoundError(TRANSLATION_CSV)
    run(TRANSLATION_CSV, minilm_output, seed=SEED)
if result_path.is_file():
    saved = json.loads(result_path.read_text(encoding="utf-8"))
    print("Resultados guardados:", result_path)
    print("Semilla:", saved["seed"], "| Traducciones:", saved["translations"])
    display(pd.DataFrame(saved["scores"]).T)
else:
    print("MiniLM omitido por RUN_MINILM.")

## Ajuste de MiniLM con el preprocesador de Alba

Comparamos las recetas de Alba por espacios y con spaCy. El texto sin preprocesar queda como referencia histórica. Los tres folds están agrupados por premisa. Chi-cuadrado puntúa las columnas con un escalado aprendido en cada fold de ajuste. Los modelos reciben las columnas seleccionadas con sus signos originales.

In [ ]:
from experiments.scripts.tune_minilm import tune

tuning_runs = [
    (ARTIFACTS / f"minilm_tuning_seed{SEED}", ("original", "spacy")),
    (ARTIFACTS / f"minilm_whitespace_seed{SEED}", ("whitespace",)),
]
tables = []
for output, inputs in tuning_runs:
    summary_path = output / "summary.csv"
    if TUNE_MINILM and not summary_path.is_file():
        tune(output, seed=SEED, inputs=inputs)
    if not TUNE_MINILM and not summary_path.is_file():
        summary_path = HISTORICAL / output.name / "summary.csv"
    if summary_path.is_file():
        tables.append(pd.read_csv(summary_path))
if tables:
    results = pd.concat(tables, ignore_index=True).sort_values(
        ["accuracy", "macro_f1"], ascending=False
    )
    alba_results = results[results["input"].isin(["spacy", "whitespace"])]
    display(alba_results.round(4))
else:
    print("Ajuste de MiniLM omitido por TUNE_MINILM.")

## PRÓXIMO

- **Bucle de Akil.** 
- **Evaluador de Akil.** 
- **Balanceo de Dario.** 